# M2 — Dense GPT
Chạy tiny config trước để xác nhận huấn luyện và sinh văn bản. Mô hình proposal-scale nằm trong `configs/dense.yaml`.


In [ ]:
REPO_URL = "https://github.com/shinbkb/mini-gpt.git"
COMMIT_SHA = "REPLACE_WITH_FULL_COMMIT_CONTAINING_M2"
M1_ARTIFACTS_INPUT = None  # Kaggle Input directory containing tokenizer.json and *_tokens.pt
CONFIG_PATH = "configs/tiny.yaml"
MAX_STEPS = 100
DEVICE = "auto"  # auto, cpu, or cuda


In [ ]:
from pathlib import Path
import os, re, shutil, subprocess, sys
root = Path('/kaggle/working/mini-gpt')
assert re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA), 'Set full M2 commit SHA after pushing it.'
if not root.exists(): subprocess.run(['git','clone',REPO_URL,str(root)], check=True)
dirty = subprocess.check_output(['git','status','--porcelain'], cwd=root, text=True)
if dirty.strip(): raise RuntimeError('Working tree has changes. Start a fresh Kaggle session.')
subprocess.run(['git','fetch','origin',COMMIT_SHA], cwd=root, check=True)
subprocess.run(['git','checkout','--detach',COMMIT_SHA], cwd=root, check=True)
actual = subprocess.check_output(['git','rev-parse','HEAD'], cwd=root, text=True).strip()
assert actual.lower() == COMMIT_SHA.lower()
os.chdir(root)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], check=True)
if M1_ARTIFACTS_INPUT:
    source = Path(M1_ARTIFACTS_INPUT)
    for name in ('tokenizer.json','train_tokens.pt','validation_tokens.pt'):
        matches = list(source.rglob(name))
        if not matches: raise FileNotFoundError(f'{name} not found below {source}')
        Path('artifacts').mkdir(exist_ok=True)
        shutil.copy2(matches[0], Path('artifacts') / name)
required = [Path('artifacts') / n for n in ('tokenizer.json','train_tokens.pt','validation_tokens.pt')]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Attach M1 outputs as a Kaggle Input or set M1_ARTIFACTS_INPUT: ' + ', '.join(missing))


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q'], check=True)
subprocess.run([sys.executable,'-m','src.environment','--seed','42'], check=True)


In [ ]:
command = [sys.executable,'train.py','--config',CONFIG_PATH,'--max-steps',str(MAX_STEPS),'--device',DEVICE]
subprocess.run(command, check=True)


In [ ]:
subprocess.run([sys.executable,'-m','src.inference.generate',
                '--checkpoint','runs/tiny/best.pt' if CONFIG_PATH == 'configs/tiny.yaml' else 'runs/dense/best.pt',
                '--tokenizer','artifacts/tokenizer.json','--prompt','Once upon a time',
                '--max-new-tokens','100','--device',DEVICE], check=True)
print('Save Version to preserve runs/, metrics and checkpoints as Kaggle Notebook Output.')
